# Identificacion de patrones de consumo energetico con aprendizaje no supervisado

En este proyecto se usa inteligencia artificial para analizar patrones de consumo electrico dentro del sistema regional PJM Interconnection.

A diferencia del proyecto anterior, aqui no se busca predecir un valor exacto de consumo. El objetivo es agrupar dias con comportamientos similares para identificar patrones de consumo normal, alta demanda, bajo consumo o posibles comportamientos atipicos.

El enfoque usado es **aprendizaje no supervisado**, porque el modelo no recibe una respuesta correcta ni una etiqueta previa. El algoritmo debe descubrir grupos ocultos directamente a partir de los datos.


## 1. Introduccion

Las empresas electricas manejan grandes cantidades de datos historicos de consumo. Estos datos pueden mostrar patrones importantes, como dias con demanda normal, dias con consumo alto, dias de bajo consumo o dias con comportamientos poco comunes.

El sistema PJM Interconnection es una organizacion regional de transmision electrica de Estados Unidos. El dataset contiene registros horarios asociados al consumo electrico dentro de este sistema.

En este notebook se prepara la informacion horaria, se resume por dia y se aplica K-Means Clustering para encontrar grupos de dias parecidos.


## 2. Problematica

Revisar manualmente miles de registros horarios de consumo electrico es dificil y poco eficiente. Cuando una empresa electrica no identifica a tiempo patrones de alta demanda o comportamientos atipicos, puede tener problemas de planeacion y operacion.

Detectar automaticamente dias de alta demanda, baja demanda o comportamiento anormal puede ayudar a:

- Planear mejor la produccion y distribucion de energia.
- Preparar recursos de respaldo.
- Analizar picos de consumo.
- Apoyar la toma de decisiones operativas.


## 3. Solucion con IA

La solucion propuesta usa aprendizaje no supervisado para agrupar dias con patrones similares de consumo energetico.

El modelo no conoce previamente si un dia fue normal, alto, bajo o atipico. En lugar de eso, analiza variables diarias como consumo promedio, consumo maximo, consumo minimo, consumo total, rango de consumo, desviacion y hora pico.

Con esas variables, K-Means separa los dias en grupos llamados clusters. Luego se interpretan esos grupos revisando los valores promedio de cada cluster.


## 4. Importar librerias

Se importan las librerias necesarias para cargar datos, preparar variables, normalizar, entrenar K-Means, evaluar los clusters y crear graficas.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)


## 5. Cargar el dataset

Se carga el archivo `dataset/pjm_hourly_est.csv`. Este archivo contiene registros historicos por hora del consumo electrico dentro del sistema PJM.

Algunas columnas representan zonas del sistema y la columna `PJM_Load` representa la carga total estimada del sistema. Para este proyecto se usa `PJM_Load` como base para construir los patrones diarios de consumo.


In [ ]:
rutas_posibles = [
    Path('dataset/pjm_hourly_est.csv'),
    Path('../dataset/pjm_hourly_est.csv')
]

ruta_dataset = next((ruta for ruta in rutas_posibles if ruta.exists()), None)

if ruta_dataset is None:
    raise FileNotFoundError('No se encontro el archivo dataset/pjm_hourly_est.csv')

df = pd.read_csv(ruta_dataset)

print(f'Archivo cargado: {ruta_dataset}')
display(df.head())


## 6. Informacion general del dataset

Antes de preparar los datos, se revisa la estructura general del archivo. Esto permite conocer columnas, tipos de datos, estadisticas basicas y valores nulos.


In [ ]:
df.info()


In [ ]:
display(df.describe())


In [ ]:
valores_nulos = pd.DataFrame({
    'Columna': df.columns,
    'Valores nulos': df.isna().sum().values,
    'Porcentaje nulos': (df.isna().mean().values * 100).round(2)
})

display(valores_nulos)


## 7. Limpieza y preparacion de datos

Para trabajar con datos horarios es necesario convertir la fecha a formato `datetime`, ordenar los registros por tiempo y asegurar que las columnas numericas esten en formato correcto.

En este proyecto se trabaja con la columna `PJM_Load`, porque representa la carga total estimada del sistema PJM. Luego se crean variables temporales y se agrupan los datos por dia.


In [ ]:
datos = df.copy()

if 'Datetime' not in datos.columns:
    raise ValueError('El dataset debe contener la columna Datetime')

if 'PJM_Load' not in datos.columns:
    raise ValueError('El dataset debe contener la columna PJM_Load')

# Se convierte la fecha y la carga total.
datos['Datetime'] = pd.to_datetime(datos['Datetime'], errors='coerce')
datos['PJM_Load'] = pd.to_numeric(datos['PJM_Load'], errors='coerce')

# Se ordenan los registros por fecha.
datos = datos.sort_values('Datetime').reset_index(drop=True)


In [ ]:
columnas_numericas = datos.select_dtypes(include='number').columns.tolist()

print('Columnas numericas encontradas:')
display(pd.DataFrame({'Columna numerica': columnas_numericas}))


In [ ]:
# Se conservan registros con fecha y consumo validos.
datos_horarios = datos[['Datetime', 'PJM_Load']].dropna().copy()

# Se crean variables temporales.
datos_horarios['fecha'] = datos_horarios['Datetime'].dt.floor('D')
datos_horarios['hora'] = datos_horarios['Datetime'].dt.hour
datos_horarios['dia'] = datos_horarios['Datetime'].dt.day
datos_horarios['mes'] = datos_horarios['Datetime'].dt.month
datos_horarios['dia_semana'] = datos_horarios['Datetime'].dt.dayofweek

print('Registros horarios validos:', len(datos_horarios))
display(datos_horarios.head())


## 8. Crear el dataset diario

El dataset original esta por hora. Para agrupar dias completos, se crea un nuevo dataset diario.

Cada fila del nuevo dataset representa un dia y resume el comportamiento de consumo de ese dia mediante variables estadisticas.


In [ ]:
resumen_diario = datos_horarios.groupby('fecha').agg(
    consumo_promedio=('PJM_Load', 'mean'),
    consumo_maximo=('PJM_Load', 'max'),
    consumo_minimo=('PJM_Load', 'min'),
    consumo_total=('PJM_Load', 'sum'),
    desviacion_consumo=('PJM_Load', 'std'),
    registros_dia=('PJM_Load', 'count')
).reset_index()

resumen_diario['rango_consumo'] = resumen_diario['consumo_maximo'] - resumen_diario['consumo_minimo']

indice_hora_pico = datos_horarios.groupby('fecha')['PJM_Load'].idxmax()
horas_pico = datos_horarios.loc[indice_hora_pico, ['fecha', 'hora']].rename(columns={'hora': 'hora_pico'})

datos_diarios = resumen_diario.merge(horas_pico, on='fecha', how='left')

# Se evitan dias con muy pocos registros.
datos_diarios = datos_diarios[datos_diarios['registros_dia'] >= 18].copy()
datos_diarios = datos_diarios.dropna().reset_index(drop=True)

print('Dias disponibles para el modelo:', len(datos_diarios))
display(datos_diarios.head())


## 9. Variables para el aprendizaje no supervisado

Para aplicar K-Means se usan variables que describen el comportamiento diario del consumo energetico.

| Variable | Que representa | Por que ayuda |
|---|---|---|
| consumo_promedio | Promedio diario de consumo | Resume el nivel general del dia |
| consumo_maximo | Mayor consumo registrado en el dia | Permite detectar picos de demanda |
| consumo_minimo | Menor consumo registrado en el dia | Ayuda a identificar momentos de baja demanda |
| consumo_total | Suma del consumo horario del dia | Representa la energia total demandada |
| rango_consumo | Diferencia entre maximo y minimo | Mide la variacion dentro del dia |
| desviacion_consumo | Dispersion del consumo horario | Ayuda a detectar dias inestables o atipicos |
| hora_pico | Hora en que ocurre el mayor consumo | Permite comparar el momento de maxima demanda |


In [ ]:
variables_cluster = [
    'consumo_promedio',
    'consumo_maximo',
    'consumo_minimo',
    'consumo_total',
    'rango_consumo',
    'desviacion_consumo',
    'hora_pico'
]

X = datos_diarios[variables_cluster].copy()

display(X.head())


## 10. Visualizacion inicial del consumo diario

Antes de entrenar el modelo, se observa el consumo promedio por dia. Esta grafica ayuda a reconocer cambios generales de demanda a lo largo del tiempo.


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(datos_diarios['fecha'], datos_diarios['consumo_promedio'], linewidth=1)
plt.title('Consumo promedio por dia')
plt.xlabel('Fecha')
plt.ylabel('Consumo promedio diario (MW)')
plt.tight_layout()
plt.show()


## 11. Normalizacion de variables

K-Means calcula distancias entre los datos. Por eso es importante normalizar las variables antes de entrenar el modelo.

Si una variable tiene valores mucho mas grandes que otra, puede dominar el calculo de distancia. Con `StandardScaler`, todas las variables quedan en una escala comparable.


In [ ]:
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(X)

X_normalizado = pd.DataFrame(X_normalizado, columns=variables_cluster)

display(X_normalizado.head())


## 12. Seleccion del numero de clusters

Para elegir un numero adecuado de clusters se usa el metodo del codo. Este metodo entrena K-Means con diferentes valores de `k` y revisa la inercia.

La inercia mide que tan cerca quedan los datos de su centro de cluster. Se busca un punto donde agregar mas clusters ya no reduzca mucho la inercia.


In [ ]:
inercias = []
valores_k = range(2, 11)

for k in valores_k:
    modelo_codo = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo_codo.fit(X_normalizado)
    inercias.append(modelo_codo.inertia_)

tabla_codo = pd.DataFrame({
    'Numero de clusters': list(valores_k),
    'Inercia': inercias
})

display(tabla_codo)


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(list(valores_k), inercias, marker='o')
plt.title('Metodo del codo')
plt.xlabel('Numero de clusters')
plt.ylabel('Inercia')
plt.xticks(list(valores_k))
plt.tight_layout()
plt.show()


## 13. Entrenamiento del modelo K-Means

Con base en el metodo del codo y para facilitar la interpretacion en clase, se usa un modelo con 4 clusters.

La idea es obtener grupos que puedan interpretarse como consumo normal, alta demanda, bajo consumo y comportamiento atipico.


In [ ]:
numero_clusters = 4

modelo_kmeans = KMeans(n_clusters=numero_clusters, random_state=42, n_init=10)
datos_diarios['Cluster'] = modelo_kmeans.fit_predict(X_normalizado)

display(datos_diarios[['fecha', 'consumo_promedio', 'consumo_maximo', 'Cluster']].head())


## 14. Evaluacion del agrupamiento

Para evaluar el agrupamiento se usa el **Silhouette Score**. Este valor ayuda a entender que tan bien separados quedaron los grupos.

El valor puede ir aproximadamente de -1 a 1. Un valor mas alto indica que los datos estan mejor agrupados y mas separados de otros clusters.


In [ ]:
silhouette = silhouette_score(X_normalizado, datos_diarios['Cluster'])

print(f'Silhouette Score: {silhouette:.4f}')


## 15. Analisis de resultados por cluster

Para interpretar los clusters, se revisan los valores promedio de cada grupo. La interpretacion no se asigna al azar: se comparan el consumo promedio, el consumo maximo y la variacion del consumo.


In [ ]:
perfil_clusters = datos_diarios.groupby('Cluster')[variables_cluster].mean().round(2)
cantidad_por_cluster = datos_diarios['Cluster'].value_counts().sort_index()

perfil_clusters['cantidad_dias'] = cantidad_por_cluster

display(perfil_clusters)


In [ ]:
orden_consumo = perfil_clusters['consumo_promedio'].sort_values()
cluster_bajo = orden_consumo.index[0]
cluster_alto = orden_consumo.index[-1]

candidatos_atipicos = perfil_clusters.drop(index=[cluster_bajo, cluster_alto], errors='ignore')

if len(candidatos_atipicos) > 0:
    cluster_atipico = candidatos_atipicos['desviacion_consumo'].idxmax()
else:
    cluster_atipico = perfil_clusters['desviacion_consumo'].idxmax()

nombres_cluster = {cluster: 'Consumo normal' for cluster in perfil_clusters.index}
nombres_cluster[cluster_bajo] = 'Bajo consumo'
nombres_cluster[cluster_alto] = 'Alta demanda'
nombres_cluster[cluster_atipico] = 'Comportamiento atipico'

datos_diarios['Nombre_cluster'] = datos_diarios['Cluster'].map(nombres_cluster)

interpretacion_clusters = perfil_clusters.copy()
interpretacion_clusters['Interpretacion'] = interpretacion_clusters.index.map(nombres_cluster)

display(interpretacion_clusters)


## 16. Interpretacion de los clusters

La tabla anterior permite explicar cada grupo de forma sencilla:

- **Alta demanda**: cluster con mayor consumo promedio diario.
- **Bajo consumo**: cluster con menor consumo promedio diario.
- **Comportamiento atipico**: cluster con alta variacion interna de consumo.
- **Consumo normal**: dias con valores intermedios y comportamiento mas estable.

Esta interpretacion puede cambiar si se entrena el modelo con nuevos datos, porque K-Means encuentra grupos segun la estructura del dataset disponible.


## 17. Grafica de dispersion por cluster

Esta grafica compara el consumo promedio contra el consumo maximo. Cada color representa un cluster asignado por K-Means.


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=datos_diarios,
    x='consumo_promedio',
    y='consumo_maximo',
    hue='Nombre_cluster',
    palette='Set2',
    alpha=0.8
)
plt.title('Consumo promedio vs consumo maximo por cluster')
plt.xlabel('Consumo promedio diario (MW)')
plt.ylabel('Consumo maximo diario (MW)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


## 18. Promedio de consumo por cluster

La siguiente grafica muestra el consumo promedio diario de cada grupo. Esto facilita identificar grupos de alta y baja demanda.


In [ ]:
promedio_cluster = datos_diarios.groupby('Nombre_cluster')['consumo_promedio'].mean().sort_values()

plt.figure(figsize=(10, 5))
sns.barplot(x=promedio_cluster.index, y=promedio_cluster.values, palette='Set2', hue=promedio_cluster.index, legend=False)
plt.title('Promedio de consumo por cluster')
plt.xlabel('Cluster')
plt.ylabel('Consumo promedio diario (MW)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 19. Cantidad de dias por cluster

Esta grafica muestra cuantos dias quedaron en cada grupo. Si un cluster tiene pocos dias, puede representar un comportamiento menos frecuente.


In [ ]:
conteo_cluster = datos_diarios['Nombre_cluster'].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=conteo_cluster.index, y=conteo_cluster.values, palette='Set3', hue=conteo_cluster.index, legend=False)
plt.title('Cantidad de dias por cluster')
plt.xlabel('Cluster')
plt.ylabel('Cantidad de dias')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 20. Serie temporal con clusters asignados

Esta grafica permite ver como cambian los grupos a lo largo del tiempo. Cada punto representa un dia y su color indica el cluster asignado.


In [ ]:
plt.figure(figsize=(14, 6))
sns.scatterplot(
    data=datos_diarios,
    x='fecha',
    y='consumo_promedio',
    hue='Nombre_cluster',
    palette='Set2',
    s=35,
    alpha=0.85
)
plt.title('Consumo diario y cluster asignado')
plt.xlabel('Fecha')
plt.ylabel('Consumo promedio diario (MW)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


## 21. Ejemplos de dias por cluster

Para entender mejor los resultados, se muestran algunos dias representativos de cada cluster.


In [ ]:
ejemplos_cluster = datos_diarios.sort_values(['Cluster', 'fecha']).groupby('Nombre_cluster').head(5)

columnas_ejemplo = [
    'fecha',
    'Nombre_cluster',
    'consumo_promedio',
    'consumo_maximo',
    'consumo_minimo',
    'rango_consumo',
    'desviacion_consumo',
    'hora_pico'
]

display(ejemplos_cluster[columnas_ejemplo])


## 22. Conclusion

El modelo logro separar los dias segun patrones de consumo energetico. Al analizar variables diarias como consumo promedio, consumo maximo, consumo minimo, consumo total, rango, desviacion y hora pico, K-Means encontro grupos con comportamientos similares.

Este enfoque puede ayudar a detectar dias de alta demanda, dias de consumo normal, dias de bajo consumo y posibles comportamientos atipicos dentro del sistema PJM.

Al ser aprendizaje no supervisado, el modelo no predice un valor exacto ni usa una respuesta correcta. Su funcion es descubrir grupos ocultos en los datos para apoyar el analisis y la toma de decisiones.


## 23. Diferencia con el proyecto anterior

En el proyecto anterior se usaba aprendizaje supervisado. Ese enfoque buscaba predecir un valor objetivo de consumo energetico usando una red neuronal.

En este nuevo proyecto se usa aprendizaje no supervisado. Aqui no se usa una variable objetivo ni etiquetas previas. El modelo agrupa datos parecidos y permite descubrir patrones de comportamiento energetico.

- **Antes**: se predecia un valor de consumo.
- **Ahora**: se agrupan dias con patrones similares.
